# Chapter 12: Attention & Transformers

The 2017 paper "Attention Is All You Need" changed everything.

RNNs process words one-by-one (slow, forgets early words).
Transformers process **all words at once** and let every word look at every other word.

```
RNN:         word1 → word2 → word3 → word4  (sequential)
Transformer: word1 ↔ word2 ↔ word3 ↔ word4  (all-to-all, parallel)
```

---
## The Core Idea: Attention

"The cat sat on the mat because **it** was tired."

What does "it" refer to? The cat? The mat? A human knows instantly. The model needs **attention** to figure this out.

Attention lets each word **ask a question** to all other words: "How relevant are you to me?"

```
"it" asks: how relevant is...
  "The"     → 0.02 (low)
  "cat"     → 0.70 (high! — it = cat)
  "sat"     → 0.05 (low)
  "on"      → 0.01 (low)
  "the"     → 0.02 (low)
  "mat"     → 0.15 (some)
  "because" → 0.03 (low)
  "it"      → 0.02 (low)
```

Then "it" updates its representation to mostly reflect "cat".

---
## Query, Key, Value: The Three Roles

Each word plays three roles simultaneously:

- **Query (Q)**: "What am I looking for?" (the question)
- **Key (K)**: "What do I contain?" (the label)
- **Value (V)**: "What information do I provide?" (the content)

Analogy — searching a library:
- **Query**: your search term ("books about cats")
- **Key**: each book's title/tags ("feline behavior", "dog training", ...)
- **Value**: the actual book content

You match your query against all keys, then read the values of the best matches.

In [ ]:
import numpy as np

np.random.seed(42)

def softmax(x):
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

# Simple sentence with 4 words, embedding dim = 4
words = ["The", "cat", "sat", "down"]
seq_len = len(words)
d_model = 4

# Pretend these are the embeddings (from Ch 11)
X = np.array([
    [0.1, 0.2, 0.0, 0.5],   # The
    [0.8, 0.1, 0.9, 0.2],   # cat
    [0.3, 0.7, 0.2, 0.8],   # sat
    [0.1, 0.5, 0.1, 0.9],   # down
])

# Weight matrices that transform embeddings into Q, K, V
# (These are LEARNED during training)
d_k = 3  # dimension of Q and K
W_Q = np.random.randn(d_model, d_k) * 0.3
W_K = np.random.randn(d_model, d_k) * 0.3
W_V = np.random.randn(d_model, d_k) * 0.3

# Transform: each word gets its own Q, K, V
Q = X @ W_Q  # what am I looking for?
K = X @ W_K  # what do I contain?
V = X @ W_V  # what info do I provide?

print("Step 1: Create Query, Key, Value for each word\n")
print(f"  X shape:     {X.shape}  (4 words × 4 dims)")
print(f"  W_Q shape:   {W_Q.shape}  (4 → 3 projection)")
print(f"  Q shape:     {Q.shape}  (4 words × 3 dims)")
print()

for i, word in enumerate(words):
    print(f"  '{word}':")
    print(f"    Q = [{', '.join(f'{v:+.3f}' for v in Q[i])}]  (what I look for)")
    print(f"    K = [{', '.join(f'{v:+.3f}' for v in K[i])}]  (what I contain)")
    print(f"    V = [{', '.join(f'{v:+.3f}' for v in V[i])}]  (my information)")

---
## Attention Score: Who Should I Pay Attention To?

Each word's Query is compared against every word's Key using dot product.

High dot product = "these two are relevant to each other."

```
score(word_i, word_j) = Query_i · Key_j
```

Then softmax turns scores into probabilities (weights that sum to 1).

In [ ]:
# Step 2: Compute attention scores
# Score = Q × K^T / sqrt(d_k)
scores = Q @ K.T / np.sqrt(d_k)

print("Step 2: Attention Scores (Q · K^T / √d_k)\n")
print(f"  Each cell: how much does ROW word attend to COLUMN word\n")
print(f"  {'':>8}", end="")
for w in words:
    print(f" {w:>8}", end="")
print()
print(f"  {'':>8}", end="")
print(f" {'─'*8}" * len(words))

for i, w in enumerate(words):
    print(f"  {w:>8}", end="")
    for j in range(len(words)):
        print(f" {scores[i,j]:>+8.3f}", end="")
    print()

print("\n  Divide by √d_k to prevent scores from getting too large.")
print(f"  √d_k = √{d_k} = {np.sqrt(d_k):.2f}")

In [ ]:
# Step 3: Softmax → attention weights
weights = softmax(scores)

print("Step 3: Softmax → Attention Weights (rows sum to 1.0)\n")
print(f"  {'':>8}", end="")
for w in words:
    print(f" {w:>8}", end="")
print("     Sum")
print(f"  {'':>8}", end="")
print(f" {'─'*8}" * len(words), end="")
print(f" {'─'*8}")

for i, w in enumerate(words):
    print(f"  {w:>8}", end="")
    for j in range(len(words)):
        # Highlight high attention
        val = weights[i,j]
        print(f" {val:>8.3f}", end="")
    print(f" {weights[i].sum():>8.3f}")

print("\n  Each row shows: how much this word attends to each other word.")
print("  Higher number = more attention paid.")

In [ ]:
# Visualize attention as a heatmap
print("Attention Heatmap (█=high attention, ░=low):\n")

blocks = " ░▒▓█"
print(f"  {'':>8}", end="")
for w in words:
    print(f" {w:>4}", end="")
print()

for i, w in enumerate(words):
    print(f"  {w:>8}", end="")
    for j in range(len(words)):
        level = int(weights[i,j] * 4.99)
        level = min(4, max(0, level))
        print(f"  {blocks[level]*2}", end="")
    print()

In [ ]:
# Step 4: Weighted sum of Values
output = weights @ V

print("Step 4: Output = Attention Weights × Values\n")
print("  Each word's new representation is a weighted mix of all Values.\n")

for i, w in enumerate(words):
    print(f"  '{w}' new representation:")
    print(f"    = ", end="")
    parts = []
    for j, w2 in enumerate(words):
        parts.append(f"{weights[i,j]:.2f}×V({w2})")
    print(" + ".join(parts))
    print(f"    = [{', '.join(f'{v:+.3f}' for v in output[i])}]")
    print()

print("Each word now contains information from all words it attended to!")
print("This is how 'it' can learn to represent 'cat'.")

---
## The Complete Attention Formula

All four steps in one equation:

```
Attention(Q, K, V) = softmax(Q × K^T / √d_k) × V
```

That's it. The entire self-attention mechanism.

In [ ]:
def self_attention(X, W_Q, W_K, W_V):
    """Complete self-attention in 4 lines."""
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    d_k = Q.shape[-1]
    return softmax(Q @ K.T / np.sqrt(d_k)) @ V

result = self_attention(X, W_Q, W_K, W_V)

print("Self-Attention in 4 Lines of Code:\n")
print("  def self_attention(X, W_Q, W_K, W_V):")
print("      Q = X @ W_Q                              # queries")
print("      K = X @ W_K                              # keys")
print("      V = X @ W_V                              # values")
print("      return softmax(Q @ K.T / √d_k) @ V      # attend & mix")
print(f"\n  Input shape:  {X.shape}  (4 words × 4 dims)")
print(f"  Output shape: {result.shape}  (4 words × {d_k} dims)")
print(f"\n  Every word now encodes context from ALL other words.")

---
## Multi-Head Attention: Multiple Perspectives

One attention head might learn "what noun does this pronoun refer to?"
Another might learn "what adjective describes this noun?"

**Multi-head attention** runs several attention operations in parallel, each learning different relationships.

```
Head 1: focuses on syntactic relationships (subject-verb)
Head 2: focuses on semantic similarity (cat-animal)
Head 3: focuses on positional patterns (next word)
...
Concatenate all heads → Linear projection → Output
```

In [ ]:
def multi_head_attention(X, n_heads, d_model, d_k):
    """Multi-head attention: multiple attention patterns in parallel."""
    heads = []
    for h in range(n_heads):
        W_Q = np.random.randn(d_model, d_k) * 0.3
        W_K = np.random.randn(d_model, d_k) * 0.3
        W_V = np.random.randn(d_model, d_k) * 0.3
        head_output = self_attention(X, W_Q, W_K, W_V)
        heads.append(head_output)
    
    # Concatenate all heads
    concat = np.concatenate(heads, axis=-1)
    
    # Project back to d_model dimensions
    W_O = np.random.randn(n_heads * d_k, d_model) * 0.3
    return concat @ W_O

np.random.seed(42)
n_heads = 2
mha_output = multi_head_attention(X, n_heads=n_heads, d_model=d_model, d_k=d_k)

print(f"Multi-Head Attention ({n_heads} heads):\n")
print(f"  Each head has its own W_Q, W_K, W_V")
print(f"  Each head learns DIFFERENT attention patterns\n")
print(f"  Head 1 output: {(X.shape[0])} × {d_k} = shape ({X.shape[0]}, {d_k})")
print(f"  Head 2 output: {(X.shape[0])} × {d_k} = shape ({X.shape[0]}, {d_k})")
print(f"  Concatenated:  {(X.shape[0])} × {n_heads * d_k} = shape ({X.shape[0]}, {n_heads * d_k})")
print(f"  Projected:     {(X.shape[0])} × {d_model} = shape {mha_output.shape}")
print(f"\n  Output is same shape as input! Can stack many layers.")

print(f"\nReal models:")
print(f"  GPT-2:    12 heads × 64 dims = 768")
print(f"  GPT-3:    96 heads × 128 dims = 12,288")
print(f"  Each head sees the sentence from a different angle.")

---
## The Transformer Block

A transformer layer combines attention with a few other components:

```
Input
  │
  ├──→ Multi-Head Attention
  │         │
  └────── + ← (residual connection, from Ch 8)
          │
       LayerNorm
          │
  ├──→ Feed-Forward Network (2 dense layers)
  │         │
  └────── + ← (residual connection)
          │
       LayerNorm
          │
       Output (same shape as input!)
```

Stack 12-96 of these blocks → that's a transformer.

In [ ]:
def layer_norm(x, eps=1e-5):
    """Normalize across features (stabilizes training)."""
    mean = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1, keepdims=True)
    return (x - mean) / (std + eps)

def feed_forward(x, W1, b1, W2, b2):
    """Two dense layers with ReLU activation."""
    hidden = np.maximum(0, x @ W1 + b1)  # ReLU
    return hidden @ W2 + b2

def transformer_block(X, n_heads, d_model, d_ff):
    """One complete transformer block."""
    d_k = d_model // n_heads
    
    # Multi-head attention + residual + norm
    attn_out = multi_head_attention(X, n_heads, d_model, d_k)
    X = layer_norm(X + attn_out)  # residual + normalize
    
    # Feed-forward + residual + norm
    W1 = np.random.randn(d_model, d_ff) * 0.1
    b1 = np.zeros((1, d_ff))
    W2 = np.random.randn(d_ff, d_model) * 0.1
    b2 = np.zeros((1, d_model))
    ff_out = feed_forward(X, W1, b1, W2, b2)
    X = layer_norm(X + ff_out)  # residual + normalize
    
    return X

np.random.seed(42)
d_model = 4
d_ff = 8  # feed-forward hidden size (usually 4× d_model)

print("One Transformer Block:\n")
print(f"  Input:  {X.shape}")
block_out = transformer_block(X, n_heads=2, d_model=d_model, d_ff=d_ff)
print(f"  Output: {block_out.shape}  (same shape!)\n")

print(f"  Input and output are the same shape.")
print(f"  This means we can stack as many blocks as we want:\n")

# Stack multiple blocks
x = X.copy()
n_layers = 6
for layer in range(n_layers):
    np.random.seed(layer)
    x = transformer_block(x, n_heads=2, d_model=d_model, d_ff=d_ff)
    print(f"  Layer {layer+1}: shape {x.shape}  norm={np.linalg.norm(x):.3f}")

print(f"\n  {n_layers} layers stacked. Each layer refines the representations.")
print(f"  Residual connections keep gradients flowing (Ch 8).")

---
## Causal Masking: Don't Look Ahead

When generating text, a word should only attend to words **before** it (not future words).

```
"The cat sat" → predict next word

"The" can see:  [The]
"cat" can see:  [The, cat]
"sat" can see:  [The, cat, sat]
"???" can see:  [The, cat, sat, ???]  ← predict this
```

This is done by **masking** (setting to -infinity) the upper triangle of the attention scores.

In [ ]:
# Causal masking
np.random.seed(42)
Q = X @ (np.random.randn(d_model, d_k) * 0.3)
K = X @ (np.random.randn(d_model, d_k) * 0.3)

scores = Q @ K.T / np.sqrt(d_k)

print("Causal (Autoregressive) Masking:\n")

print("Before masking (all words see all words):")
print(f"  {'':>8}", end="")
for w in words:
    print(f" {w:>6}", end="")
print()
for i, w in enumerate(words):
    print(f"  {w:>8}", end="")
    for j in range(len(words)):
        print(f" {scores[i,j]:>+6.2f}", end="")
    print()

# Apply causal mask
mask = np.triu(np.ones((seq_len, seq_len)), k=1)  # upper triangle
masked_scores = scores - mask * 1e9  # -inf for future positions

print("\nAfter masking (can only see past and self):")
print(f"  {'':>8}", end="")
for w in words:
    print(f" {w:>6}", end="")
print()
for i, w in enumerate(words):
    print(f"  {w:>8}", end="")
    for j in range(len(words)):
        if j > i:
            print(f"   -inf", end="")
        else:
            print(f" {scores[i,j]:>+6.2f}", end="")
    print()

masked_weights = softmax(masked_scores)
print("\nAttention weights after masking:")
print(f"  {'':>8}", end="")
for w in words:
    print(f" {w:>6}", end="")
print()
for i, w in enumerate(words):
    print(f"  {w:>8}", end="")
    for j in range(len(words)):
        print(f" {masked_weights[i,j]:>6.3f}", end="")
    print()

print("\n  'The'  only attends to itself (first word).")
print("  'down' attends to all 4 words (last word).")
print("  No word can peek at the future!")

---
## Transformer Sizes

Real transformers are the same architecture, just scaled up massively.

In [ ]:
print("Transformer Model Sizes:\n")
print(f"  {'Model':<18} {'Layers':>7} {'Heads':>6} {'d_model':>8} {'Params':>10}")
print(f"  {'─'*18} {'─'*7} {'─'*6} {'─'*8} {'─'*10}")

models = [
    ("Our example",       1,    2,     4,     "~100"),
    ("BERT-base",        12,   12,   768,    "110M"),
    ("GPT-2",            12,   12,   768,    "117M"),
    ("BERT-large",       24,   16,  1024,    "340M"),
    ("GPT-2 XL",         48,   25,  1600,    "1.5B"),
    ("GPT-3",            96,   96, 12288,    "175B"),
    ("LLaMA-70B",        80,   64,  8192,     "70B"),
]

for name, layers, heads, d, params in models:
    print(f"  {name:<18} {layers:>7} {heads:>6} {d:>8} {params:>10}")

print("\n  Same architecture at every scale.")
print("  GPT-3 is 'just' 96 transformer blocks stacked.")
print("  The magic is in the SCALE + TRAINING DATA.")

---
## Why Transformers Won

```
RNN:          Sequential → slow, forgets early tokens
Transformer:  Parallel   → fast on GPUs, sees everything
```

In [ ]:
print("Why Transformers Replaced RNNs:\n")

comparisons = [
    ("Processing",    "Sequential (word by word)",       "Parallel (all at once)"),
    ("GPU utilization", "Poor (can't parallelize)",      "Excellent (matrix multiply)"),
    ("Long-range",    "Fades over distance",             "Direct connection (attention)"),
    ("Training speed", "Slow (sequential dependency)",   "Fast (parallelizable)"),
    ("Scaling",       "Hard to scale beyond ~1B",        "Scales to 100B+ parameters"),
    ("Context",       "Fixed hidden state size",         "Grows with sequence (O(n²))"),
]

print(f"  {'Aspect':<18} {'RNN/LSTM':<32} {'Transformer'}")
print(f"  {'─'*18} {'─'*32} {'─'*32}")
for aspect, rnn, transformer in comparisons:
    print(f"  {aspect:<18} {rnn:<32} {transformer}")

print("\n  The O(n²) cost of attention is the main drawback:")
print("    1K tokens:   1M operations    (fast)")
print("    10K tokens:  100M operations  (manageable)")
print("    100K tokens: 10B operations   (expensive!)")
print("  This is why context windows have limits.")

---
## Summary

| Concept | Key Point |
|---------|----------|
| **Attention** | Every word looks at every other word |
| **Q, K, V** | Query (question), Key (label), Value (content) |
| **Score** | Q · K^T / √d_k → how relevant are two words |
| **Softmax** | Turns scores into weights (sum to 1) |
| **Multi-head** | Multiple attention patterns in parallel |
| **Transformer block** | Attention + Feed-forward + Residual + LayerNorm |
| **Causal mask** | Prevents looking at future words (for generation) |
| **O(n²)** | Cost grows quadratically with sequence length |

**Next chapter**: Large Language Models — putting it all together